# Limpieza de datos

La mayoría de la gente piensa que ser científico de datos significa estar corriendo modelos avanzados de Machine Learning todo el tiempo. La realidad es muy distinta:

![Tiempo de un científico de datos](./img/datascientist_time.jpeg)

La gran mayoría del tiempo se va **limpiando y organizando** los datos con los que queremos trabajar.

Los científicos de datos utilizamos herramientas como Pandas, Numpy, Matplotlib y Seaborn para limpiar los datos con los que queremos hacer algo.

## ETL

Un tipo de tarea que realizamos con gran frecuencia los científicos de datos son los **ETL**.

![Proceso de ETL](./img/etl.png)

ETL es un acrónimo que significa Extract, Transform, Load. Es un proceso que se utiliza para extraer datos de una fuente, transformarlos en un formato que sea adecuado para el análisis y cargarlos en una base de datos o algún otro sistema de almacenamiento.

## Ejemplo práctico

Los datos que usaremos para esta limpieza y nuestro siguiente análisis son datos de incidencia delictiva en nuestro país.

La iniciativa de datos abiertos del gobierno de México nos proporciona datos de incidencia delictiva desde 2015 hasta la fecha. Los datos se actualizan todos los meses y se pueden descargar desde el siguiente enlace: https://www.gob.mx/sesnsp/acciones-y-programas/datos-abiertos-de-incidencia-delictiva

---

Como podemos ver en el portal, se proporcionan los datos tanto a nivel estatal como a nivel municipal. En este caso, utilizaremos los datos a nivel estatal.

Descarguemos los datos y guardemos el archivo CSV en la carpeta data con el nombre `datos_delitos.csv`.

---

Ahora leamos el archivo CSV y veamos cómo se ven los datos.

Primero que nada, importemos pandas

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data/datos_delitos.csv')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf1 in position 1: invalid continuation byte

Aquí tenemos un error muy común que suele ocurrir cuando un archivo se guarda en una computadora con cierto "encoding". 

Un encoding es una tabla que relaciona un número con un carácter. Por ejemplo, en la tabla ASCII, el número 65 corresponde a la letra "A".

Si el archivo que estamos leyendo fue guardado con un encoding distinto al que pandas espera, nos arrojará un error. 

Para solucionar esto, podemos utilizar el parámetro `encoding` de la función `pd.read_csv()` y especificar el encoding correcto.

¿Pero cómo sabemos con qué encoding cuenta el archivo?

La realidad es que la gran mayoría de los archivos los encontrarán en encoding utf-8 y Pandas no va a dar ningún error. Aquí estamos teniendo este problema porque la computadora que utilizan para generar este archivo uso un encoding diferente a utf-8.

En México, por lo general, si un archivo no está en utf-8, lo más seguro es que esté en `ISO-8859-1` o `latin1`.

Intentemos cargar el archivo especificando el encoding `ISO-8859-1`.

In [3]:
df = pd.read_csv('data/datos_delitos.csv', encoding='ISO-8859-1')
df.head()

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0
2,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,0,2,2,3,2,0.0,1.0,2.0,0.0,0.0,0.0
3,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,2,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0


Y vemos que ya podemos leer correctamente el archivo.


¿Existe alguna forma de verificar el encoding de un archivo sin tener que estar adivinando?

ChatGTP generó el siguiente código:

In [4]:
import chardet

def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        rawdata = f.read()
    result = chardet.detect(rawdata)
    return result

file_path = './data/datos_delitos.csv'
encoding_info = detect_encoding(file_path)
print(f"Detected encoding: {encoding_info['encoding']}")

Detected encoding: ISO-8859-1


Sigamos...

In [5]:
df.tail(3)

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
31357,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,20,22,34,35,34,37,NaN,NaN,NaN,NaN,NaN,NaN
31358,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Electorales,Electorales,Electorales,0,1,0,11,31,23,NaN,NaN,NaN,NaN,NaN,NaN
31359,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Otros delitos del Fuero Común,Otros delitos del Fuero Común,Otros delitos del Fuero Común,151,183,174,199,208,177,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31360 entries, 0 to 31359
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Año                     31360 non-null  int64  
 1   Clave_Ent               31360 non-null  int64  
 2   Entidad                 31360 non-null  str    
 3   Bien jurídico afectado  31360 non-null  str    
 4   Tipo de delito          31360 non-null  str    
 5   Subtipo de delito       31360 non-null  str    
 6   Modalidad               31360 non-null  str    
 7   Enero                   31360 non-null  int64  
 8   Febrero                 31360 non-null  int64  
 9   Marzo                   31360 non-null  int64  
 10  Abril                   31360 non-null  int64  
 11  Mayo                    31360 non-null  int64  
 12  Junio                   31360 non-null  int64  
 13  Julio                   28224 non-null  float64
 14  Agosto                  28224 non-null  float64
 

Bien. Están muy bien los datos. Sin embargo, tenemos un problema con el que es muy común encontrarnos. 

Resulta que el formato en el que está el archivo es fácil entender por seres humanos:


```markdown
| Año      | Entidad  | Enero    | Febrero  | Mes X    |
|----------|----------|----------|----------|----------|
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
```

Vemos que tenemos los meses como encabezados. Es decir, el archivo está tratando cada mes como si fuera una variable. 

Nosotros como científicos de datos estamos más interesados en conjuntos de datos que no estén en este formato de "resumen" o "tabla dinámica". Para nosotros, lo ideal sería que cada mes fuera simplemente una observación más en nuestro conjunto de datos. Es decir, queremos transformar la tabla de arriba en:

```markdown
| Año      | Entidad  | Mes        |
|----------|----------|------------|
|   ...    |   ...    |   Enero    |
|   ...    |   ...    |   Febrero  |
|   ...    |   ...    |   Marzo    |
|   ...    |   ...    |   Abril    |
|   ...    |   ...    |   Mes X    |
```

A este tipo de formato le llamos "formato largo de datos". 

Haremos las siguientes limpiezas:
* Transformar los nombres de las columnas para que no tengan caracteres especiales y estén siempre en minúsculas
* Convertir el dataset a un formato de datos "largo"

In [7]:
for col in df.columns:
    print(col)

Año
Clave_Ent
Entidad
Bien jurídico afectado
Tipo de delito
Subtipo de delito
Modalidad
Enero
Febrero
Marzo
Abril
Mayo
Junio
Julio
Agosto
Septiembre
Octubre
Noviembre
Diciembre


In [8]:
len(df.columns)

19

### Limpiar nombres de columnas

In [9]:
def limpiar_columnas(df):
    columnas_limpias = []
    for col in df.columns:
        # convertir a minusculas, reemplazar espacios por guiones bajos y eliminar caracteres especiales
        col = col.lower().replace(" ", "_").replace("ñ", "ni").replace(".", "").replace("á", "a").replace("é", "e").replace("í","i").replace("ó", "o").replace("ú", "u")
        columnas_limpias.append(col)
    
    df.columns = columnas_limpias
    
    return df


In [10]:
df.head(2)

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0


In [11]:
df = limpiar_columnas(df)

In [12]:
df.head(3)

,anio,clave_ent,entidad,bien_juridico_afectado,tipo_de_delito,subtipo_de_delito,modalidad,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0
2,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,0,2,2,3,2,0.0,1.0,2.0,0.0,0.0,0.0


Muy bien. Ahora lo que queremos hacer es quitar algunas columnas. Nos interesan nada más las siguientes:

In [13]:
df[['anio', 'entidad']]

,anio,entidad
0,2015,Aguascalientes
1,2015,Aguascalientes
2,2015,Aguascalientes
3,2015,Aguascalientes
4,2015,Aguascalientes
...,...,...
31355,2024,Zacatecas
31356,2024,Zacatecas
31357,2024,Zacatecas
31358,2024,Zacatecas


In [14]:
df = df[['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'subtipo_de_delito', 'modalidad','enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']]
df.head(2)

,anio,clave_ent,entidad,tipo_de_delito,subtipo_de_delito,modalidad,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0


### Formato largo de datos

Ahora, usaremos el método `melt` para convertir las columnas a observaciones.

Queremos convervar las coumnas:
* anio
* clave_ent
* entidad
* tipo_de_delito
* subtipo_de_delito
* modalidad

El resto de las columnas las vamos a juntar en una nueva columna llamada "nombre_mes" y sus valores los vamos a reunir en otra llamada "frecuencia"

In [15]:
print("Shape ", df.shape)

Shape  (31360, 18)


In [16]:
datos_long = df.melt(id_vars=['anio', 'clave_ent', 'entidad','tipo_de_delito', 'subtipo_de_delito', 'modalidad'], var_name='nombre_mes', value_name='frecuencia')

# Excluir meses sin datos; un valor faltante no representa cero delitos.
datos_long = datos_long.dropna(subset=['frecuencia']).copy()
datos_long['frecuencia'] = datos_long['frecuencia'].astype('int64')


### Meses observados y valores faltantes

El archivo contiene todos los meses de 2015 a 2023 y solo enero a junio de 2024. Las 18,816 celdas sin observación de julio a diciembre de 2024 se excluyen antes de agrupar, para evitar convertirlas en ceros. Quedan 357,504 registros mensuales de detalle; la agrupación por entidad y tipo de delito genera 145,920 registros.

`melt` reorganiza los valores en filas; la suma de subtipos y modalidades ocurre después, con `groupby`. El archivo agregado pierde la distinción entre homicidio doloso y culposo y entre subtipos de robo. Para responder preguntas sobre esos subtipos se debe volver al archivo original, como hace el notebook 3.

In [17]:
print("Shape: ", datos_long.shape)

Shape:  (357504, 8)


In [18]:
datos_long.head(5)

,anio,clave_ent,entidad,tipo_de_delito,subtipo_de_delito,modalidad,nombre_mes,frecuencia
0,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma de fuego,enero,3
1,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma blanca,enero,1
2,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con otro elemento,enero,0
3,2015,1,Aguascalientes,Homicidio,Homicidio doloso,No especificado,enero,2
4,2015,1,Aguascalientes,Homicidio,Homicidio culposo,Con arma de fuego,enero,0


In [19]:
datos_long.info()

<class 'pandas.DataFrame'>
Index: 357504 entries, 0 to 373183
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   anio               357504 non-null  int64
 1   clave_ent          357504 non-null  int64
 2   entidad            357504 non-null  str  
 3   tipo_de_delito     357504 non-null  str  
 4   subtipo_de_delito  357504 non-null  str  
 5   modalidad          357504 non-null  str  
 6   nombre_mes         357504 non-null  str  
 7   frecuencia         357504 non-null  int64
dtypes: int64(3), str(5)
memory usage: 24.5 MB


Supongamos que para este análisis, no nos importan los niveles subtipo de delito y modalidad. O sea, no queremos tener la distinción entre homicidios dolosos y culposos (sé que son bastante diferentes, pero simplifiquemos nuestro ejemplo).

Vamos a agrupar nuestro dataframe por anio, clave_ent, entidad, tipo_de_delito y nombre_mes. Esto hará que todos los tipos de homicidios se sumen al tipo "homicidio" o todos los tipos de robo de vehículo (con o sin violencia) se sumen a "robo de vehículo".

In [20]:
datos_long = datos_long.groupby(['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes'])['frecuencia'].sum(min_count=1).reset_index()

In [21]:
datos_long[datos_long.tipo_de_delito == 'Robo'].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
80551,2020,8,Chihuahua,Robo,marzo,1238
875,2015,2,Baja California,Robo,septiembre,4106
6152,2015,13,Hidalgo,Robo,mayo,900
118473,2022,23,Quintana Roo,Robo,noviembre,1234
129507,2023,14,Jalisco,Robo,enero,3927
95915,2021,8,Chihuahua,Robo,septiembre,1240
14787,2015,31,Yucatán,Robo,enero,349
21995,2016,14,Jalisco,Robo,septiembre,5930
143712,2024,23,Quintana Roo,Robo,abril,1088
109350,2022,4,Campeche,Robo,junio,504


Mostremos todos los estados y su respectiva clave

In [22]:
datos_long[['clave_ent', 'entidad']].drop_duplicates().sort_values('entidad')

,clave_ent,entidad
0,1,Aguascalientes
480,2,Baja California
960,3,Baja California Sur
1440,4,Campeche
2880,7,Chiapas
3360,8,Chihuahua
3840,9,Ciudad de México
1920,5,Coahuila de Zaragoza
2400,6,Colima
4320,10,Durango


Ahora Veamos todos los datos de delitos de una entidad en específico. Por ejemplo, Nuevo león

In [23]:
datos_long[datos_long['clave_ent'] == 19].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
8778,2015,19,Nuevo León,Electorales,junio,45
24256,2016,19,Nuevo León,Incumplimiento de obligaciones de asistencia f...,febrero,38
54749,2018,19,Nuevo León,Abuso sexual,julio,113
39602,2017,19,Nuevo León,Incesto,diciembre,0
100860,2021,19,Nuevo León,Amenazas,abril,454
131911,2023,19,Nuevo León,Robo,marzo,1075
8831,2015,19,Nuevo León,Falsificación,septiembre,69
24367,2016,19,Nuevo León,Otros delitos que atentan contra la vida y la ...,marzo,2
9002,2015,19,Nuevo León,Otros delitos que atentan contra la vida y la ...,diciembre,1
116345,2022,19,Nuevo León,Falsificación,julio,128


In [24]:
datos_long[(datos_long['clave_ent'] > 19) &  (datos_long['clave_ent'] < 24) & (datos_long['tipo_de_delito'] == 'Homicidio')].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
71745,2019,22,Querétaro,Homicidio,noviembre,46
143389,2024,22,Querétaro,Homicidio,enero,30
26139,2016,23,Quintana Roo,Homicidio,enero,22
55907,2018,21,Puebla,Homicidio,septiembre,160
86145,2020,20,Oaxaca,Homicidio,noviembre,146
40061,2017,20,Oaxaca,Homicidio,julio,150
40067,2017,20,Oaxaca,Homicidio,septiembre,148
132225,2023,20,Oaxaca,Homicidio,noviembre,162
41020,2017,22,Querétaro,Homicidio,febrero,39
72216,2019,23,Quintana Roo,Homicidio,abril,136


### Valores de fechas

Finalmente, queremos tener una columna "fecha". Actualmente tenemos el año y el nombre del mes, pero no tenemos como tal una columna que tenga un tipo de dato fecha. Eso hace que filtrar por fecha sea complicado.

Por ejemplo, si queremos conocer todos los homicidios de Oaxaca en enero 2024, haríamos lo siguiente:

In [25]:
datos_long[
    (datos_long['clave_ent'] == 20) &
    (datos_long['tipo_de_delito'] == 'Homicidio') &
    (datos_long['anio'] == 2024) &
    (datos_long['nombre_mes'] == 'enero') 
]

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
142909,2024,20,Oaxaca,Homicidio,enero,157


Creemos una columna de fecha. 

Primero tenemos que convertir el nombre de mes a un número, en donde 1 es enero, 2 febrero, etc.

In [26]:
datos_long.sample(2)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
85560,2020,19,Nuevo León,Despojo,abril,33
112213,2022,10,Durango,Rapto,agosto,0


In [27]:
datos_long["nueva_columna"] = "dato vacío"
datos_long.sample(6)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna
31345,2017,2,Baja California,Evasión de presos,agosto,0,dato vacío
28179,2016,27,Tabasco,Otros delitos que atentan contra la libertad p...,enero,32,dato vacío
23812,2016,18,Nayarit,Otros delitos contra el patrimonio,febrero,0,dato vacío
33385,2017,6,Colima,Lesiones,agosto,133,dato vacío
111573,2022,9,Ciudad de México,Fraude,noviembre,1909,dato vacío
82536,2020,12,Guerrero,Violencia de género en todas sus modalidades d...,abril,14,dato vacío


In [28]:
# Diccionario de ayuda para convertir
meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

datos_long['mes'] = datos_long['nombre_mes'].map(meses)
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes
59627,2018,29,Tlaxcala,Daño a la propiedad,septiembre,23,dato vacío,9
34651,2017,9,Ciudad de México,Corrupción de menores,marzo,5,dato vacío,3
132444,2023,20,Oaxaca,Violación simple,abril,51,dato vacío,4
100244,2021,17,Morelos,Secuestro,mayo,0,dato vacío,5
76954,2020,1,Aguascalientes,Evasión de presos,octubre,0,dato vacío,10


In [29]:
datos_long["frecuencia_mas_10"] = datos_long["frecuencia"] + 10
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10
127346,2023,10,Durango,Evasión de presos,diciembre,0,dato vacío,12,10
80178,2020,8,Chihuahua,Abuso de confianza,junio,83,dato vacío,6,93
19440,2016,9,Ciudad de México,Incesto,abril,0,dato vacío,4,10
92830,2021,2,Baja California,Falsificación,octubre,36,dato vacío,10,46
20523,2016,11,Guanajuato,Otros delitos que atentan contra la vida y la ...,enero,0,dato vacío,1,10


In [30]:
datos_long["anio_mes"] = datos_long["anio"].astype(str) + datos_long["mes"].astype(str)
datos_long.sample(6)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10,anio_mes
19090,2016,8,Chihuahua,Otros delitos que atentan contra la vida y la ...,octubre,22,dato vacío,10,32,201610
27940,2016,27,Tabasco,Daño a la propiedad,febrero,221,dato vacío,2,231,20162
81578,2020,10,Durango,Violencia de género en todas sus modalidades d...,diciembre,0,dato vacío,12,10,202012
89222,2020,26,Sonora,Tráfico de menores,diciembre,0,dato vacío,12,10,202012
24223,2016,19,Nuevo León,Homicidio,marzo,91,dato vacío,3,101,20163
117910,2022,22,Querétaro,Otros delitos contra la familia,octubre,21,dato vacío,10,31,202210


In [31]:
# yyyy-mm-dd, yy-mm-dd, yymmdd, yyyy/dd/mm

In [32]:
# Agregamos la columna de fecha juntando el año y el mes
datos_long['fecha'] = pd.to_datetime(datos_long['anio'].astype(str) + datos_long['mes'].astype(str), format='%Y%m')
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10,anio_mes,fecha
118865,2022,24,San Luis Potosí,Otros delitos contra la familia,julio,4,dato vacío,7,14,20227,2022-07-01
127681,2023,11,Guanajuato,Aborto,agosto,5,dato vacío,8,15,20238,2023-08-01
41643,2017,23,Quintana Roo,Otros delitos que atentan contra la vida y la ...,enero,0,dato vacío,1,10,20171,2017-01-01
110827,2022,7,Chiapas,Tráfico de menores,marzo,0,dato vacío,3,10,20223,2022-03-01
46599,2018,2,Baja California,Acoso sexual,enero,0,dato vacío,1,10,20181,2018-01-01


In [33]:
# Eliminamos las columnas que ya no necesitamos
datos_long = datos_long.drop(columns=['nueva_columna'])
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,frecuencia_mas_10,anio_mes,fecha
127579,2023,10,Durango,Rapto,marzo,0,3,10,20233,2023-03-01
66332,2019,11,Guanajuato,Corrupción de menores,mayo,6,5,16,20195,2019-05-01
103363,2021,24,San Luis Potosí,Extorsión,marzo,5,3,15,20213,2021-03-01
2425,2015,6,Colima,Abuso sexual,agosto,0,8,10,20158,2015-08-01
83303,2020,14,Jalisco,Incumplimiento de obligaciones de asistencia f...,septiembre,0,9,10,20209,2020-09-01


Veamos los homicidios en oaxaca de enero 2024 a la fecha

In [34]:
datos_long[
    (datos_long.tipo_de_delito == "Homicidio") &
    (datos_long.clave_ent == 20) &
    (datos_long.fecha >= '2024-01-01')
].sort_values('fecha')

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,frecuencia_mas_10,anio_mes,fecha
142909,2024,20,Oaxaca,Homicidio,enero,157,1,167,20241,2024-01-01
142910,2024,20,Oaxaca,Homicidio,febrero,170,2,180,20242,2024-02-01
142912,2024,20,Oaxaca,Homicidio,marzo,190,3,200,20243,2024-03-01
142908,2024,20,Oaxaca,Homicidio,abril,208,4,218,20244,2024-04-01
142913,2024,20,Oaxaca,Homicidio,mayo,236,5,246,20245,2024-05-01
142911,2024,20,Oaxaca,Homicidio,junio,202,6,212,20246,2024-06-01


In [35]:
datos_finales = datos_long[['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes', 'fecha', 'frecuencia']]
datos_finales.head(2)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,fecha,frecuencia
0,2015,1,Aguascalientes,Aborto,abril,2015-04-01,0
1,2015,1,Aguascalientes,Aborto,agosto,2015-08-01,0


Ya que tenemos muestros datos bien estructurados, los podemos guardar en nuestra computadora. Los guardaremos con el nombre "delitos.csv"

In [36]:
datos_finales.to_csv('data/delitos.csv', index=False)

## Resultados de la limpieza

- El archivo original contiene 31,360 filas y 19 columnas. La lectura inicial con UTF-8 produce el error de codificación que explica la lección; se conserva como ejemplo intencional y la siguiente lectura usa `ISO-8859-1`.
- Los nombres de columnas se normalizan y los meses se convierten en observaciones con una fecha.
- Se exportan 145,920 filas y 7 columnas a `data/delitos.csv`, con cobertura observada del 1 de enero de 2015 al 1 de junio de 2024 (la fecha identifica el mes).
- Los datos son estatales: no contienen municipios. Las frecuencias son registros de incidencia del archivo; no se reinterpretan como número de víctimas.
- El total de 2024 corresponde únicamente a enero–junio y no es comparable directamente con un año completo.
